# 00 — CARLA headless setup on Google Colab

CARLA is not officially supported on Colab, but the server can run headless (no display) using off-screen rendering. This notebook:

1. Downloads and extracts the CARLA prebuilt Linux server (must match the `carla` client version pinned in `requirements.txt`).
2. Launches it in the background with `-RenderOffScreen` (no X server needed).
3. Waits for the RPC port to come up and runs a 10-second smoke test (spawn one vehicle, tick, destroy).

**If step 3 fails**, the most common cause is Colab's GPU type not supporting the Vulkan renderer CARLA defaults to. Try re-running the launch cell with `-opengl` appended (see the fallback cell near the bottom) before assuming something else is wrong. A second common cause is picking a Colab runtime without a GPU at all — check *Runtime > Change runtime type > GPU* first.

Run this notebook first, once per Colab session (the server does not survive a runtime restart).

In [ ]:
!nvidia-smi

In [ ]:
CARLA_VERSION = "0.9.15"
CARLA_TAR_URL = f"https://carla-releases.s3.us-east-005.backblazeb2.com/Linux/CARLA_{CARLA_VERSION}.tar.gz"  # the old tiny.carla.org short-link is dead; this is CARLA's actual release storage bucket
CARLA_DIR = "/content/carla_server"

import os
os.makedirs(CARLA_DIR, exist_ok=True)

In [ ]:
%%bash -s "$CARLA_TAR_URL" "$CARLA_DIR"
set -e
cd "$2"
if [ ! -f CarlaUE4.sh ]; then
  echo "Downloading CARLA server (this is large, ~15-20GB — grab a coffee)..."
  wget -q --show-progress -O carla.tar.gz "$1"
  tar -xzf carla.tar.gz
  rm carla.tar.gz
else
  echo "CARLA server already extracted, skipping download."
fi
ls

## Install a Python 3.10 environment for the CARLA client

CARLA 0.9.15's Python client library only has wheels for Python 3.7–3.10, but Colab's own Python is newer than that (3.10+ / 3.13). Rather than fight Colab's main Python, this creates a separate Python 3.10 virtual environment just for anything that does `import carla`. Every CARLA-dependent step from here on runs through that venv's interpreter (`/content/carla_venv/bin/python`) via a small script in `scripts/`, instead of running interactively inside this notebook's own Python.

This cell only needs to run once per Colab session; it's slower the first time (installing Python 3.10 + packages) but fast on a re-run.

In [ ]:
%%bash
set -e
if [ ! -x /content/carla_venv/bin/python ]; then
  apt-get -qq update
  apt-get -qq install -y software-properties-common ca-certificates > /dev/null
  add-apt-repository -y ppa:deadsnakes/ppa > /dev/null
  apt-get -qq update
  apt-get -qq install -y python3.10 python3.10-venv python3.10-dev > /dev/null
  python3.10 -m venv /content/carla_venv
  /content/carla_venv/bin/python -m ensurepip --upgrade > /dev/null
  /content/carla_venv/bin/python -m pip install --upgrade pip -q
  /content/carla_venv/bin/pip install -q carla==0.9.15 numpy pandas scipy matplotlib gymnasium stable-baselines3
  echo "venv created"
else
  echo "venv already exists, skipping"
fi
/content/carla_venv/bin/python --version
/content/carla_venv/bin/python -c "import carla; print('carla import OK, version', carla.__file__)"

## Launch the server headless (background process)

`-RenderOffScreen`: no display required. `-carla-server`: RPC server. `-nosound`: skip audio init (not available in a headless container).

Re-running this cell after a crash is safe — it kills any previous instance first.

In [ ]:
import subprocess
import time

subprocess.run(["pkill", "-f", "CarlaUE4"], check=False)
time.sleep(2)

carla_process = subprocess.Popen(
    [f"{CARLA_DIR}/CarlaUE4.sh", "-RenderOffScreen", "-carla-server", "-nosound", "-quality-level=Low"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print("Launching CARLA server, pid:", carla_process.pid)
time.sleep(20)  # first boot is slow; increase if the connection cell below times out

## Fallback: if the cell above's server never accepts connections, try `-opengl`

Some Colab GPU types don't have a usable Vulkan ICD inside the container. Uncomment and run this cell INSTEAD of the one above if the smoke test below keeps timing out.

In [ ]:
# subprocess.run(["pkill", "-f", "CarlaUE4"], check=False)
# time.sleep(2)
# carla_process = subprocess.Popen(
#     [f"{CARLA_DIR}/CarlaUE4.sh", "-RenderOffScreen", "-opengl", "-carla-server", "-nosound", "-quality-level=Low"],
#     stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
# )
# print("Launching CARLA server (opengl fallback), pid:", carla_process.pid)
# time.sleep(20)

## Smoke test: connect, spawn one vehicle, tick, destroy

In [ ]:
!/content/carla_venv/bin/python /content/fag-project/scripts/smoke_test.py

## Calibrate the merge point (one-time, per CARLA version)

`scenario.py`'s auto-detection picks the first junction with >=2 converging driving lanes, which may not be the highway on-ramp you want. Print the candidates and their coordinates here; if the first one is wrong, hardcode the right `carla.Transform` into `MANUAL_MERGE_POINT` in `src/merge_sim/scenario.py`.

In [ ]:
!/content/carla_venv/bin/python /content/fag-project/scripts/calibrate_merge_point.py